<a href="https://colab.research.google.com/github/mdrs-thiago/mdrs-thiago.github.io/blob/main/topicos_a/2027/notebooks/lab05_finetuning_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Lab 5 — Fine-tuning de BERT
**Tópicos Especiais A · Deep Learning · UERJ 2027.1** · Aula 20 · Bloco 5 — Transformers

**Objetivo:** fazer fine-tuning de um Transformer no mesmo split do Lab 4 e comparar acurácia, custo e latência.

| | |
|---|---|
| ✅ **Checkpoint (em sala)** | 60 XP: veja as marcações `✅ CHECKPOINT` |
| 📦 **Entrega (7 dias)** | 180 XP: tabela final dos 5 modelos, 10 exemplos BERT × LSTM e recomendação para produção. |

> Preencha apenas os trechos marcados com `# TODO`. Antes de começar: **Arquivo → Salvar uma cópia no Drive** e, se indicado, **Ambiente de execução → Alterar tipo → T4 GPU**.

In [ ]:
!pip -q install datasets evaluate

In [ ]:
import random, numpy as np, torch
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)

## 1. Mesmo split do Lab 4
Faça upload do `split_lab4.json` (painel de arquivos à esquerda).

In [ ]:
import json, time
from datasets import load_dataset, Dataset
imdb = load_dataset('imdb')
split = json.load(open('split_lab4.json'))
train_ds = imdb['train'].select(split['train'])
val_ds = imdb['test'].select(split['val'])
print(train_ds, val_ds)

## 2. Baseline zero-shot (modelo de sentimento já pronto, sem fine-tuning seu)

In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score
zs = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english',
              device=0 if device == 'cuda' else -1, truncation=True, max_length=256)
t0 = time.time()
preds = [1 if r['label'] == 'POSITIVE' else 0 for r in zs(val_ds['text'], batch_size=64)]
res = [{'modelo': 'zero-shot (SST-2)', 'acc': accuracy_score(val_ds['label'], preds), 'f1': f1_score(val_ds['label'], preds), 'tempo treino (s)': 0}]
print(res, f'{time.time() - t0:.0f}s de inferência')

## 3. Tokenização

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
MODEL = 'distilbert-base-uncased'   # se usou PT no Lab 4: 'neuralmind/bert-base-portuguese-cased'
tok = AutoTokenizer.from_pretrained(MODEL)
def tokenize(b):
    # TODO: tokenize b['text'] com truncation=True, max_length=256 (padding dinâmico fica com o Trainer)
    return ...
train_tok = train_ds.map(tokenize, batched=True)
val_tok = val_ds.map(tokenize, batched=True)

## 4. Fine-tuning com `Trainer`
### ✅ CHECKPOINT: treino iniciado, com a avaliação da primeira época aparecendo.

In [ ]:
import evaluate, numpy as np
from transformers import TrainingArguments, Trainer
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
acc_m, f1_m = evaluate.load('accuracy'), evaluate.load('f1')
def metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {**acc_m.compute(predictions=preds, references=p.label_ids), **f1_m.compute(predictions=preds, references=p.label_ids)}

args = TrainingArguments(
    output_dir='out', num_train_epochs=2, per_device_train_batch_size=16, per_device_eval_batch_size=64,
    # TODO: learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    fp16=(device == 'cuda'), report_to='none', seed=42)
trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok,
                  processing_class=tok, compute_metrics=metrics)

torch.cuda.reset_peak_memory_stats() if device == 'cuda' else None
t0 = time.time(); trainer.train(); t_train = time.time() - t0
ev = trainer.evaluate()
peak = torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else float('nan')
res.append({'modelo': f'{MODEL} fine-tunado', 'acc': ev['eval_accuracy'], 'f1': ev['eval_f1'], 'tempo treino (s)': round(t_train)})
print(f'pico de memória: {peak:.1f} GB')

## 5. Latência de inferência

In [ ]:
def latency(model, device_, n=50):
    model.to(device_).eval()
    enc = tok(val_ds[0]['text'], return_tensors='pt', truncation=True, max_length=256).to(device_)
    with torch.no_grad():
        for _ in range(5): model(**enc)
        t0 = time.time()
        for _ in range(n): model(**enc)
    return (time.time() - t0) / n * 1000
# TODO: meça em 'cpu' e (se houver) em 'cuda'; anote junto com os números da LSTM do Lab 4
print('CPU ms/exemplo:', latency(trainer.model, 'cpu', n=10))

## 6. Tabela final
Copie os resultados do Lab 4 (TF-IDF, BiLSTM, BiLSTM + atenção) para a mesma tabela.

In [ ]:
import pandas as pd
lab4 = [  # TODO: preencha com os seus números do Lab 4
    {'modelo': 'TF-IDF + LogReg', 'acc': None, 'f1': None, 'tempo treino (s)': None},
    {'modelo': 'BiLSTM', 'acc': None, 'f1': None, 'tempo treino (s)': None},
    {'modelo': 'BiLSTM + atenção', 'acc': None, 'f1': None, 'tempo treino (s)': None},
]
pd.DataFrame(lab4 + res)

## 7. Análise de erros: BERT × LSTM

In [ ]:
p_bert = np.argmax(trainer.predict(val_tok).predictions, axis=-1)
# TODO: carregue as predições da LSTM do Lab 4 (salve-as lá com np.save) e compare
# p_lstm = np.load('preds_lstm.npy')
# idx = np.where((p_bert == np.array(val_ds['label'])) & (p_lstm != np.array(val_ds['label'])))[0][:10]
# for i in idx: print(val_ds[int(i)]['text'][:400], '\n---')

## 8. Análise (📦 entrega)
- Quanto o fine-tuning ganhou sobre a LSTM e sobre o zero-shot? A que custo (tempo, memória, latência)?
- Que tipo de review o BERT acerta e a LSTM erra? (ironia? negação longa? vocabulário raro?)
- **Recomendação:** qual modelo você colocaria em produção, e sob qual restrição?
- (Opcional) `trainer.push_to_hub()` com um model card.